# Piškorky algoritmus minmax

In [34]:
import numpy as np
import time

In [39]:
class State:
    generated = 0

    def __init__(self, gameplan, player, current_player=None, depth=0, max_depth=3):
        self.gameplan = gameplan
        self.player = player
        self.current_player = current_player if current_player else player
        self.depth = depth
        self.max_depth = max_depth
        State.generated += 1

    def terminal_test(self):
        for i in range(3):
            if np.array_equal(self.gameplan[i], [1, 1, 1]) or np.array_equal(self.gameplan[:, i], [1, 1, 1]):
                return 1
            if np.array_equal(self.gameplan[i], [2, 2, 2]) or np.array_equal(self.gameplan[:, i], [2, 2, 2]):
                return 2

        if np.array_equal(self.gameplan.diagonal(), [1, 1, 1]) or \
           np.array_equal(np.fliplr(self.gameplan).diagonal(), [1, 1, 1]):
            return 1
        if np.array_equal(self.gameplan.diagonal(), [2, 2, 2]) or \
           np.array_equal(np.fliplr(self.gameplan).diagonal(), [2, 2, 2]):
            return 2

        return 0

    def utility(self, result):
        if result == 0:
            return 0
        return 1 if result == self.player else -1

    def possible_actions(self):
        return [(i, j) for i in range(3) for j in range(3) if self.gameplan[i][j] == 0]

    def expand(self, select_action):
        if select_action is None:
            return None
        if select_action[0] not in range(3) or select_action[1] not in range(3):
            return None
        if self.gameplan[select_action[0], select_action[1]] != 0:
            return None

        new_array = np.copy(self.gameplan)
        new_array[select_action[0], select_action[1]] = self.current_player
        return State(new_array, self.player, self.next_current_player(), self.depth + 1, self.max_depth)

    def minmax(self, strategy="max"):
        result = self.terminal_test()
        if result != 0:
            return self.utility(result), None

        if self.depth >= self.max_depth:
            return 0, self.possible_actions()[0] if self.possible_actions() else None

        actions = self.possible_actions()
        if not actions:
            return 0, None

        selected_action = actions[0]
        if strategy == "max":
            selected_utilization_value = float('-inf')
            next_strategy = "min"
        else:
            selected_utilization_value = float('inf')
            next_strategy = "max"

        for action in actions:
            expanded_state = self.expand(action)
            if expanded_state is None:
                continue

            result = expanded_state.terminal_test()
            if result != 0:
                utilization = expanded_state.utility(result)
            else:
                if len(expanded_state.possible_actions()) == 0:
                    utilization = 0
                else:
                    utilization, _ = expanded_state.minmax(next_strategy)

            if strategy == "max" and utilization > selected_utilization_value:
                selected_utilization_value = utilization
                selected_action = action
            elif strategy == "min" and utilization < selected_utilization_value:
                selected_utilization_value = utilization
                selected_action = action

        return selected_utilization_value, selected_action

    def next_current_player(self):
        return 3 - self.current_player

    def next_player(self):
        return 3 - self.player

# Partie piškvorek

In [40]:
# Vytvoření počátečního stavu partie
    # herní plán je prázdní
    # na tahu je hrač 1
state = State(gameplan=np.array([[0, 0, 0], [0, 0, 0], [0, 0, 0]]),
              player=1, max_depth=3)

In [41]:
# Vypsání počátečního stavu
print(state.gameplan)

[[0 0 0]
 [0 0 0]
 [0 0 0]]


In [42]:
# Sledujte čas a počty generovaných stavů s postupující partií

# Cyklus pro hru, kde hrají proti sobě dvě kopie algoritmu
while True:
    # Kontrola, zda není partie u konce
    game_result = state.terminal_test()
    if game_result != 0:
        print(f"Winner is {game_result} ")
        break

    # Kontrola, zda není remíza
    if len(state.possible_actions()) == 0:
        print("Drawn")
        break

    # tah hráče
    print(f"=====================\nPlayer {state.player}")
    _, player_action = state.minmax("max")
    print(f"Select action: {player_action}")
    state = state.expand(player_action)
    print(state.gameplan)
    print(f"Generated states {State.generated}.")
    State.generated = 0

    # přepnutí partie na druhého hráče
    state.player = state.next_player()

Player 1
Select action: (0, 0)
[[1 0 0]
 [0 0 0]
 [0 0 0]]
Generated states 587.
Player 2
Select action: (0, 1)
[[1 2 0]
 [0 0 0]
 [0 0 0]]
Generated states 65.
Player 1
Select action: (0, 2)
[[1 2 1]
 [0 0 0]
 [0 0 0]]
Generated states 8.
Player 2
Select action: (1, 0)
[[1 2 1]
 [2 0 0]
 [0 0 0]]
Generated states 1.
Player 1
Select action: (1, 1)
[[1 2 1]
 [2 1 0]
 [0 0 0]]
Generated states 1.
Player 2
Select action: (1, 2)
[[1 2 1]
 [2 1 2]
 [0 0 0]]
Generated states 1.
Player 1
Select action: (2, 0)
[[1 2 1]
 [2 1 2]
 [1 0 0]]
Generated states 1.
Winner is 1 


# Úkol
- Do algoritmu přidejte omezeni na maximalní prohledávanou hloubku.
- Zkoušejte různé omezení prohledávání do hloubky.
- Sledujte, jak se mění časy a počty vygenerovaných stavů
- Mění se výsledky hry?

In [ ]:
#Čím větší je proměnná max_depth, tím déle hra trvá, generuje se více tahů a výsledky her se mění.

In [ ]:
def run_game_for_depth(max_depth):
    initial_state = np.array([[0, 0, 0], [0, 0, 0], [0, 0, 0]])

    start_time = time.time()

    state = State(gameplan=np.copy(initial_state), player=1, max_depth=max_depth)
    move_count = 0

    while True:
        game_result = state.terminal_test()
        if game_result != 0:
            break

        if len(state.possible_actions()) == 0:
            break

        _, player_action = state.minmax("max")
        state = state.expand(player_action)
        state.player = state.next_player()
        move_count += 1

    end_time = time.time()
    elapsed_time = round(end_time - start_time, 2)

    if game_result == 1:
        result = "Player 1 wins"
    elif game_result == 2:
        result = "Player 2 wins"
    else:
        result = "Draw"

    return move_count, elapsed_time, result, State.generated


for max_depth in [2, 5, 10, 15, 20]:
    print(f"\n=====================\nTest for max_depth = {max_depth}")
    
    move_count, elapsed_time, result, generated_states = run_game_for_depth(max_depth)
    
    print(f"Generated states: {generated_states}")
    print(f"Game time: {elapsed_time} seconds")
    print(f"Result: {result}")
    print(f"Moves made: {move_count}")
    print(f"=====================\n")


Test for max_depth = 2
Generated states: 644232
Game time: 0.02 seconds
Result: Player 1 wins
Moves made: 7


Test for max_depth = 5
Generated states: 665349
Game time: 3.32 seconds
Result: Player 1 wins
Moves made: 7


Test for max_depth = 10
Generated states: 1283534
Game time: 66.9 seconds
Result: Draw
Moves made: 9


Test for max_depth = 15
Generated states: 1901719
Game time: 66.79 seconds
Result: Draw
Moves made: 9


Test for max_depth = 20
Generated states: 2519904
Game time: 66.72 seconds
Result: Draw
Moves made: 9

